In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# S5 — Uniform Attention with a Decoupled Output Head (Appendix D re-run)

Appendix D crossed attention type with `model_type in {spherical_wd, spherical_nowd}`. In that
code the bounded head is tied to `spherical_nowd`, so the two spherical rows were
**sphere/raw at WD=1** and **sphere/bounded at WD=0** — both of which already fail on S5 under
learned attention. The one spherical configuration that solves S5 (**sphere/bounded at WD=1**,
7/10 seeds) was never reachable, so Appendix D's "topology, not routing" inference was never
actually tested.

This notebook decouples `residual geometry`, `head geometry`, `weight decay`, and `attention`
so that cell can be run. Conditions are ordered critical-first: if you stop early, you still
have the one that matters.

| # | Residual | Head | WD | Attention | Why |
|---|---|---|---|---|---|
| 1 | sphere | bounded | 1 | **uniform** | the missing cell |
| 2 | sphere | bounded | 1 | learned | matched control (paper: 7/10) |
| 3 | layernorm | raw | 1 | uniform | Appendix D baseline, matched logging |
| 4 | layernorm | raw | 1 | learned | grokking reference |

Matched to `S5_Final.ipynb`: p=S5, 30% train, d_model=128, 4 heads, MLP 512, AdamW
betas=(0.9, 0.999), LR=1e-3, 100k epochs. Seeds 1-5. Same seed gives the same split everywhere.

Completed runs are skipped; interrupted runs resume from their recovery checkpoint.

In [ ]:
import os, json, math, random, time, itertools
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Matched to S5_Final.ipynb.
N_DEGREE = 5
GROUP_SIZE = math.factorial(N_DEGREE)   # 120
FRAC_TRAIN = 0.30
D_MODEL = 128
NUM_HEADS = 4
MLP_DIM = 512
TAU = 10.0
LR = 1e-3
BETAS = (0.9, 0.999)                    # S5_Final value, NOT the 0.98 used in the addition ablation
MAX_EPOCHS = 100_000
LOG_EVERY = 100
RECOVERY_EVERY = 2_000
POST_99_GRACE = 2_000                   # keep training a while after Test-99 before stopping
STABLE_99_LOGS = 3
SEEDS = [1, 2, 3, 4, 5]

SAVE_ROOT = Path('/content/drive/MyDrive/grokking_rebuttal_s5_uniform_attention_v1')
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

# Critical-first ordering.
CONDITIONS = [
    dict(
        name="sphere_bounded_wd1_uniform",
        model_type="spherical",
        bounded_head=True,
        weight_decay=1.0,
        uniform_attention=True,
    ),
]
#CONDITIONS = [
#    dict(name='sphere_bounded_wd1_uniform',   model_type='spherical', bounded_head=True,
#         weight_decay=1.0, uniform_attention=True),
#    dict(name='sphere_bounded_wd1_learned',   model_type='spherical', bounded_head=True,
#         weight_decay=1.0, uniform_attention=False),
#    dict(name='layernorm_raw_wd1_uniform',    model_type='layernorm', bounded_head=False,
#         weight_decay=1.0, uniform_attention=True),
#    dict(name='layernorm_raw_wd1_learned',    model_type='layernorm', bounded_head=False,
#         weight_decay=1.0, uniform_attention=False),
#]

print(f'{len(CONDITIONS)} conditions x {len(SEEDS)} seeds = {len(CONDITIONS) * len(SEEDS)} runs')

In [ ]:
def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def compose_s5(id1, id2, id_to_elem, elem_to_id):
    """Apply p2 then p1 — identical to S5_Final."""
    p1, p2 = id_to_elem[id1], id_to_elem[id2]
    return elem_to_id[tuple(p1[p2[i]] for i in range(N_DEGREE))]


def make_s5_dataset(frac_train, seed):
    elements = list(itertools.permutations(range(N_DEGREE)))
    elem_to_id = {e: i for i, e in enumerate(elements)}
    id_to_elem = {i: e for i, e in enumerate(elements)}

    all_pairs = [(a, b) for a in range(GROUP_SIZE) for b in range(GROUP_SIZE)]
    random.Random(seed).shuffle(all_pairs)
    n_train = int(len(all_pairs) * frac_train)
    op = GROUP_SIZE

    def build(part):
        x = torch.tensor([[a, b, op] for a, b in part], dtype=torch.long)
        y = torch.tensor([compose_s5(a, b, id_to_elem, elem_to_id) for a, b in part],
                         dtype=torch.long)
        return x, y

    train_x, train_y = build(all_pairs[:n_train])
    test_x, test_y = build(all_pairs[n_train:])
    return train_x, train_y, test_x, test_y


class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        return x * torch.rsqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps) * self.weight


class S5Transformer(nn.Module):
    """Same architecture as S5_Final, with head geometry decoupled from weight decay."""

    def __init__(self, vocab_size, num_classes, model_type, bounded_head, uniform_attention):
        super().__init__()
        self.model_type = model_type
        self.bounded_head = bounded_head
        self.uniform_attention = uniform_attention
        self.d_head = D_MODEL // NUM_HEADS

        self.tok_embed = nn.Embedding(vocab_size, D_MODEL)
        self.pos_embed = nn.Embedding(3, D_MODEL)
        self.W_Q = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.W_K = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.W_V = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.W_O = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.mlp_in = nn.Linear(D_MODEL, MLP_DIM)
        self.mlp_out = nn.Linear(MLP_DIM, D_MODEL)
        self.unembed = nn.Linear(D_MODEL, num_classes, bias=False)

        if model_type == 'layernorm':
            self.norm1, self.norm2, self.norm3 = (nn.LayerNorm(D_MODEL) for _ in range(3))
        elif model_type == 'rmsnorm':
            self.norm1, self.norm2, self.norm3 = (RMSNorm(D_MODEL) for _ in range(3))

    def apply_norm(self, x, norm_module):
        if self.model_type in ('layernorm', 'rmsnorm'):
            return norm_module(x)
        return F.normalize(x, dim=-1)

    def forward(self, x):
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0)
        h = self.tok_embed(x) + self.pos_embed(pos)
        h = self.apply_norm(h, getattr(self, 'norm1', None))

        Q = self.W_Q(h).view(B, L, NUM_HEADS, self.d_head).transpose(1, 2)
        K = self.W_K(h).view(B, L, NUM_HEADS, self.d_head).transpose(1, 2)
        V = self.W_V(h).view(B, L, NUM_HEADS, self.d_head).transpose(1, 2)

        if self.uniform_attention:
            # Intervention B: zero pre-softmax scores -> uniform [1/3, 1/3, 1/3].
            scores = torch.zeros((B, NUM_HEADS, L, L), device=x.device, dtype=Q.dtype)
        else:
            scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_head)

        attn_out = self.W_O(
            (F.softmax(scores, dim=-1) @ V).transpose(1, 2).contiguous().view(B, L, D_MODEL)
        )
        h = self.apply_norm(h + attn_out, getattr(self, 'norm2', None))
        h = h + self.mlp_out(F.relu(self.mlp_in(h)))
        h = self.apply_norm(h, getattr(self, 'norm3', None))
        h_final = h[:, 2, :]

        if self.bounded_head:
            return F.linear(h_final, F.normalize(self.unembed.weight, dim=1)) * TAU
        return self.unembed(h_final)

In [ ]:
def atomic_torch_save(obj, path):
    tmp = Path(str(path) + '.tmp')
    torch.save(obj, tmp)
    os.replace(tmp, path)


def first_epoch(history, key, threshold):
    for e, v in zip(history['epoch'], history[key]):
        if v >= threshold:
            return int(e)
    return None


def test_rise_onset(history):
    """First logged test acc >=10% that gains >=5 points within 500 epochs (S5 timescale)."""
    e = np.asarray(history['epoch'])
    a = np.asarray(history['test_acc'])
    for i in range(len(e)):
        if a[i] < 0.10:
            continue
        fut = (e > e[i]) & (e <= e[i] + 500)
        if fut.any() and a[fut].max() >= a[i] + 0.05:
            return int(e[i])
    return None


@torch.no_grad()
def evaluate(model, x, y, criterion):
    model.eval()
    logits = model(x)
    return criterion(logits, y).item(), (logits.argmax(-1) == y).float().mean().item()


def train_one(cfg, seed, paths):
    train_x, train_y, test_x, test_y = make_s5_dataset(FRAC_TRAIN, seed)
    train_x, train_y = train_x.to(DEVICE), train_y.to(DEVICE)
    test_x, test_y = test_x.to(DEVICE), test_y.to(DEVICE)

    set_seed(seed)
    model = S5Transformer(GROUP_SIZE + 1, GROUP_SIZE, cfg['model_type'],
                          cfg['bounded_head'], cfg['uniform_attention']).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR,
                                  weight_decay=cfg['weight_decay'], betas=BETAS)
    criterion = nn.CrossEntropyLoss()

    history = {k: [] for k in ['epoch', 'train_loss', 'test_loss', 'train_acc', 'test_acc']}
    start_epoch, test99_epoch, stable99 = 0, None, 0
    stop_reason = 'max_epochs'

    if paths['recovery'].exists():
        state = torch.load(paths['recovery'], map_location=DEVICE)
        model.load_state_dict(state['model_state_dict'])
        optimizer.load_state_dict(state['optimizer_state_dict'])
        history = state['history']
        start_epoch = state['epoch'] + 1
        test99_epoch = state.get('test99_epoch')
        stable99 = state.get('stable99', 0)
        print('resume:', cfg['name'], 'seed', seed, 'from epoch', start_epoch)

    bar = tqdm(range(start_epoch, MAX_EPOCHS), desc=f"{cfg['name']} seed{seed}")
    started = time.time()
    for epoch in bar:
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(train_x), train_y)
        if not torch.isfinite(loss):
            stop_reason = f'non_finite_loss_epoch_{epoch}'
            break
        loss.backward()
        optimizer.step()

        if epoch % LOG_EVERY == 0 or epoch == MAX_EPOCHS - 1:
            tr_loss, tr_acc = evaluate(model, train_x, train_y, criterion)
            te_loss, te_acc = evaluate(model, test_x, test_y, criterion)
            for k, v in [('epoch', epoch), ('train_loss', tr_loss), ('test_loss', te_loss),
                         ('train_acc', tr_acc), ('test_acc', te_acc)]:
                history[k].append(v)
            bar.set_postfix(train=f'{tr_acc:.3f}', test=f'{te_acc:.3f}')

            if test99_epoch is None and te_acc >= 0.99:
                test99_epoch = epoch
            stable99 = stable99 + 1 if te_acc >= 0.99 else 0
            if stable99 >= STABLE_99_LOGS and test99_epoch is not None \
                    and epoch >= test99_epoch + POST_99_GRACE:
                stop_reason = 'stable_test_99_post_grace'
                break

        if epoch % RECOVERY_EVERY == 0:
            atomic_torch_save(dict(epoch=epoch, model_state_dict=model.state_dict(),
                                   optimizer_state_dict=optimizer.state_dict(),
                                   history=history, test99_epoch=test99_epoch,
                                   stable99=stable99), paths['recovery'])

    train99 = first_epoch(history, 'train_acc', 0.99)
    test99 = first_epoch(history, 'test_acc', 0.99)
    onset = test_rise_onset(history)
    summary = dict(
        condition=cfg['name'], model_type=cfg['model_type'],
        head_type='bounded' if cfg['bounded_head'] else 'raw',
        attention='uniform' if cfg['uniform_attention'] else 'learned',
        weight_decay=cfg['weight_decay'], lr=LR, seed=seed,
        train_epoch_95=first_epoch(history, 'train_acc', 0.95),
        train_epoch_99=train99,
        test_epoch_95=first_epoch(history, 'test_acc', 0.95),
        test_epoch_99=test99,
        test_rise_onset=onset,
        gap_99=None if (train99 is None or test99 is None) else test99 - train99,
        onset_minus_train_99=None if (train99 is None or onset is None) else onset - train99,
        peak_test_acc=max(history['test_acc']),
        final_train_acc=history['train_acc'][-1],
        last_epoch=history['epoch'][-1],
        stop_reason=stop_reason,
        minutes=round((time.time() - started) / 60, 1),
    )
    atomic_torch_save(model.state_dict(), paths['model'])
    with open(paths['json'], 'w') as f:
        json.dump(dict(config={k: v for k, v in cfg.items()}, summary=summary,
                       history=history), f)
    if paths['recovery'].exists():
        paths['recovery'].unlink()
    return summary

In [ ]:
rows = []
for cfg in CONDITIONS:                 # condition-first: finish the critical cell before moving on
    run_dir = SAVE_ROOT / cfg['name']
    run_dir.mkdir(parents=True, exist_ok=True)
    for seed in SEEDS:
        stem = f"{cfg['name']}__seed{seed}"
        paths = {'json': run_dir / f'{stem}.json',
                 'model': run_dir / f'{stem}.pt',
                 'recovery': run_dir / f'{stem}_recovery.pt'}

        if paths['json'].exists() and paths['model'].exists():
            summary = json.load(open(paths['json']))['summary']
            print('skip completed:', stem)
        else:
            summary = train_one(cfg, seed, paths)
            print('saved:', stem, '| Test99:', summary['test_epoch_99'],
                  '| peak:', f"{summary['peak_test_acc']:.4f}",
                  '| final train:', f"{summary['final_train_acc']:.4f}")

        rows.append(summary)
        pd.DataFrame(rows).to_csv(SAVE_ROOT / 'summary_incremental.csv', index=False)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv(SAVE_ROOT / 'per_run_results.csv', index=False)
print('\nPER-RUN')
print(df[['condition', 'attention', 'seed', 'train_epoch_99', 'test_rise_onset',
          'test_epoch_99', 'gap_99', 'peak_test_acc', 'final_train_acc',
          'stop_reason']].to_string(index=False))

In [ ]:
summary = (
    df.groupby(['condition', 'model_type', 'head_type', 'attention', 'weight_decay'],
               dropna=False)
      .agg(seeds=('seed', 'count'),
           reached_99=('test_epoch_99', lambda x: int(x.notna().sum())),
           mean_test_99=('test_epoch_99', 'mean'),
           median_test_99=('test_epoch_99', 'median'),
           std_test_99=('test_epoch_99', 'std'),
           mean_gap_99=('gap_99', 'mean'),
           mean_onset_minus_train_99=('onset_minus_train_99', 'mean'),
           mean_peak_test=('peak_test_acc', 'mean'),
           max_peak_test=('peak_test_acc', 'max'),
           mean_final_train=('final_train_acc', 'mean'))
      .reset_index()
)
summary.to_csv(SAVE_ROOT / 's5_uniform_summary.csv', index=False)
print(summary.to_string(index=False))

print("""
How to read this:

  sphere/bounded/WD=1, uniform  vs  sphere/bounded/WD=1, learned
    -> the comparison Appendix D was missing. If uniform attention does not rescue the
       spherical model here either, the "topology, not routing" reading is supported on
       the one cell that can actually test it. If it does rescue it, that inference goes.

  Every run reaching Test-99 after Train-99 (gap_99 > 0) means the run still groks, i.e.
  no pre-memorization test rise of the kind measured on addition.
""")